In [0]:
%pip install requests

In [0]:
import requests
import json


API_KEY = "e0d9d069-c5e9-4397-97c6-6f3f7d79220e"

company_number = "00002065"

base_url = f"https://api.company-information.service.gov.uk/company/{company_number}"

response = requests.get(base_url, auth=(API_KEY, ""))

data = response.json()

print(data)

In [0]:
import requests
import time
import json
import os

API_KEY = "e0d9d069-c5e9-4397-97c6-6f3f7d79220e"

company_numbers = [
    "04366849",
    "00617987",
    "00102498",
    "00041424",
    "01833679",
    "00048839",
    "02723534",
    "02231246",
    "00502851",
    "01372603",
    "01106260",
    "02366640",
    "02366619",
    "00714275",
    "00395826",
    "05562053",
    "03263464",
    "03033654",
    "04627044",
    "00185647"
]

# Base output directory (change this as needed)
BASE_PATH = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/company_house"

# Create folder structure
folders = {
    "overview": os.path.join(BASE_PATH, "overview"),
    "filing_history": os.path.join(BASE_PATH, "filing_history"),
    "people": os.path.join(BASE_PATH, "people")
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)


def fetch_and_save(url, file_path):
    try:
        response = requests.get(url, auth=(API_KEY, ""))
        
        if response.status_code == 200:
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(response.json(), f, indent=2)
        else:
            print(f"Failed ({response.status_code}): {url}")
    
    except Exception as e:
        print(f"Error calling API: {e}")


for c in company_numbers:
    print(f"Processing company: {c}")

    # -------- Overview --------
    overview_url = f"https://api.company-information.service.gov.uk/company/{c}"
    fetch_and_save(
        overview_url,
        os.path.join(folders["overview"], f"{c}.json")
    )

    # -------- Filing History --------
    filing_url = f"https://api.company-information.service.gov.uk/company/{c}/filing-history"
    fetch_and_save(
        filing_url,
        os.path.join(folders["filing_history"], f"{c}.json")
    )

    # --------  Officers --------
    people_url = f"https://api.company-information.service.gov.uk/company/{c}/officers"
    fetch_and_save(
        people_url,
        os.path.join(folders["people"], f"{c}.json")
    )

    # Rate limiting (Companies House recommends this)
    time.sleep(0.5)

print("Data ingestion complete")